# 04 — Point-Event Tabular Baselines

This notebook builds leakage-controlled physical-only, cyber-only, and fused tabular baselines from Notebook 03 outputs.

## Target policy

The preferred target is `cyber_attack_window_label` for the primary ±120-second window. With dense NetFlow data, however, every event window can contain at least one attack-labeled flow, making that target single-class and unsuitable for binary modeling. When that happens, this notebook automatically derives a **high NF3 attack-burden target** from the chronological training partition only. It uses the first variable target source among:

1. `nf3_attack_flow_fraction`
2. `nf3_attack_flow_count`

The training-only threshold is selected to approximate the upper quartile while preserving both training classes. The chosen source, operator, threshold, and observed prevalence are saved in `model_target_definition_v4.json` and the build summary. This fallback is a controlled experimental target and is not evidence of a real PANC cyberattack or causal aircraft–network interaction.


Version 3 adds a deterministic nearest-flow target fallback. When the window-level any-attack target is saturated, the notebook selects the primary-window flow whose start time is nearest to the aligned event time and uses that flow's preserved NF3 binary label. Labels remain target-only.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import re
import sys
import time
import warnings
from dataclasses import dataclass
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any, Iterable, cast

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

RANDOM_STATE = 42
MODEL_VERSION = "point_event_tabular_baselines_leakage_safe_v4"
PRIMARY_WINDOW_NAME = "primary_120s"
RAW_TARGET_COLUMN = "cyber_attack_window_label"
TARGET_COLUMN = "model_target"
TIME_COLUMN = "aircraft_event_time_utc"
GROUP_COLUMN = "aircraft_key"
ID_COLUMN = "event_id"

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15
HIGH_BURDEN_DESIRED_POSITIVE_FRACTION = 0.25
HIGH_BURDEN_MIN_CLASS_ROWS = 2

RUN_GROUPED_SENSITIVITY = True
GROUPED_SENSITIVITY_FEATURE_SETS = {"fused"}
GROUPED_SENSITIVITY_MODELS = {"logistic_regression", "random_forest"}
RUN_PERMUTATION_IMPORTANCE = False
PERMUTATION_SAMPLE_MAX = 2_000
PERMUTATION_REPEATS = 5
LATENCY_EVENT_SAMPLE_MAX = 100
RANDOM_FOREST_TREES = 200

# Absolute calendar fields are excluded by default because controlled replay can
# make physical clock time correlate with the replayed NF3 timeline.
INCLUDE_CALENDAR_FEATURES = False

PROJECT_ROOT = Path.cwd()
ALIGNMENT_DIR = PROJECT_ROOT / "data" / "processed" / "point_event_v2" / "nf3_alignment_v2"
FEATURES_IN = ALIGNMENT_DIR / "candidate_point_events_v2_with_context_and_nf3_primary.csv"
TARGETS_IN = ALIGNMENT_DIR / "event_nf3_targets_v2.csv"
ALIGNMENT_SUMMARY_IN = ALIGNMENT_DIR / "point_event_nf3_alignment_leakage_safe_v2_build_summary.json"
EDGES_IN = ALIGNMENT_DIR / "event_flow_edges_primary_bounded_v2.csv"
FLOW_LABELS_IN = ALIGNMENT_DIR / "nf3_flow_labels_used_v2.csv.gz"

# Version 3 is written separately because it adds a deterministic nearest-flow
# target fallback for saturated window-level targets.
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "point_event_v2" / "modeling_v4"
MODELS_DIR = OUTPUT_DIR / "models"
FIGURES_DIR = OUTPUT_DIR / "figures"

SPLITS_OUT = OUTPUT_DIR / "model_split_assignments_v4.csv"
RESULTS_OUT = OUTPUT_DIR / "baseline_model_results_v4.csv"
PREDICTIONS_OUT = OUTPUT_DIR / "baseline_test_predictions_v4.csv"
THRESHOLDS_OUT = OUTPUT_DIR / "baseline_thresholds_v4.json"
TARGET_DEFINITION_OUT = OUTPUT_DIR / "model_target_definition_v4.json"
LOGISTIC_COEFFICIENTS_OUT = OUTPUT_DIR / "logistic_coefficients_v4.csv"
TREE_IMPORTANCE_OUT = OUTPUT_DIR / "tree_feature_importance_v4.csv"
PERMUTATION_IMPORTANCE_OUT = OUTPUT_DIR / "permutation_importance_v4.csv"
LATENCY_OUT = OUTPUT_DIR / "baseline_latency_results_v4.csv"
FUSED_GAIN_OUT = OUTPUT_DIR / "fused_gain_summary_v4.csv"
FEATURE_MANIFEST_OUT = OUTPUT_DIR / "feature_manifest_v4.json"
SUMMARY_OUT = OUTPUT_DIR / "baseline_modeling_v4_build_summary.json"
DATA_DICTIONARY_OUT = OUTPUT_DIR / "baseline_modeling_v4_data_dictionary.csv"
CHECKSUMS_OUT = OUTPUT_DIR / "SHA256SUMS.txt"

for directory in [OUTPUT_DIR, MODELS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Feature input: {FEATURES_IN}")
print(f"Target input:  {TARGETS_IN}")
print(f"Output folder: {OUTPUT_DIR}")


## 1. Validation and conversion helpers

In [ ]:
def require_int(value: object, field_name: str) -> int:
    """Convert a required scalar to a Python int without accepting missing values."""
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")

    if isinstance(value, bool):
        return 1 if value else 0

    text = value.decode("utf-8", errors="strict") if isinstance(value, bytes) else str(value)
    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{field_name} cannot be missing.")

    numeric = float(text)
    if not math.isfinite(numeric) or not numeric.is_integer():
        raise ValueError(f"{field_name} must be a finite whole number; received {value!r}.")
    return int(numeric)


def require_float(value: object, field_name: str) -> float:
    """Convert a required scalar to a finite Python float."""
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")

    text = value.decode("utf-8", errors="strict") if isinstance(value, bytes) else str(value)
    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{field_name} cannot be missing.")

    numeric = float(text)
    if not math.isfinite(numeric):
        raise ValueError(f"{field_name} must be finite; received {value!r}.")
    return numeric


def optional_float(value: object) -> float:
    """Convert a scalar to float, returning NaN for missing/non-finite values."""
    try:
        return require_float(value, "optional numeric value")
    except (TypeError, ValueError):
        return float("nan")


def require_text(value: object, field_name: str) -> str:
    """Convert a required scalar identifier to a clean string."""
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")
    text = value.decode("utf-8", errors="strict") if isinstance(value, bytes) else str(value)
    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{field_name} cannot be empty or missing.")
    return text


def require_utc_timestamp(value: object, field_name: str) -> pd.Timestamp:
    """Convert a required scalar to a timezone-aware UTC Timestamp."""
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{field_name} cannot be missing.")

    if isinstance(value, pd.Timestamp):
        timestamp = value
    elif isinstance(value, (datetime, date, np.datetime64, str, int, float)):
        timestamp = pd.Timestamp(value)
    elif isinstance(value, bytes):
        timestamp = pd.Timestamp(value.decode("utf-8", errors="strict"))
    else:
        timestamp = pd.Timestamp(str(value))

    if pd.isna(timestamp):
        raise ValueError(f"{field_name} cannot be missing.")
    if timestamp.tzinfo is None:
        timestamp = timestamp.tz_localize("UTC")
    else:
        timestamp = timestamp.tz_convert("UTC")
    return timestamp


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def make_one_hot_encoder() -> OneHotEncoder:
    """Create a version-compatible one-hot encoder."""
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            min_frequency=2,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
        )


def has_both_classes(values: pd.Series | np.ndarray) -> bool:
    observed = set(
        pd.to_numeric(pd.Series(values), errors="coerce")
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    return observed == {0, 1}


def time_only_chronological_assignments(data: pd.DataFrame) -> pd.Series:
    """Create deterministic 70/15/15 time-based assignments without using labels."""
    ordered = data.sort_values([TIME_COLUMN, ID_COLUMN], kind="mergesort")
    row_count = len(ordered)
    if row_count < 20:
        raise ValueError("At least 20 events are required for chronological train/validation/test splits.")

    train_position = max(1, min(row_count - 3, int(math.floor(row_count * TRAIN_FRACTION))))
    validation_end_position = max(
        train_position + 1,
        min(row_count - 1, int(math.floor(row_count * (TRAIN_FRACTION + VALIDATION_FRACTION)))),
    )

    train_boundary = require_utc_timestamp(
        ordered.iloc[train_position - 1][TIME_COLUMN],
        "chronological train boundary",
    )
    validation_boundary = require_utc_timestamp(
        ordered.iloc[validation_end_position - 1][TIME_COLUMN],
        "chronological validation boundary",
    )

    assignment = pd.Series("test", index=data.index, dtype="string")
    assignment.loc[data[TIME_COLUMN].le(train_boundary)] = "train"
    assignment.loc[
        data[TIME_COLUMN].gt(train_boundary)
        & data[TIME_COLUMN].le(validation_boundary)
    ] = "validation"
    return assignment


def choose_training_burden_threshold(
    values: pd.Series,
    desired_positive_fraction: float,
    minimum_class_rows: int,
) -> tuple[float, float]:
    """Choose a training-only threshold with prevalence nearest the requested fraction."""
    numeric = pd.to_numeric(values, errors="coerce").dropna().astype(float)
    if numeric.empty or numeric.nunique() < 2:
        raise ValueError("Target source does not vary in the chronological training partition.")

    unique_values = np.sort(numeric.unique())
    candidates: list[tuple[float, float, float]] = []
    for threshold in unique_values[1:]:
        positive_rows = int(numeric.ge(threshold).sum())
        negative_rows = int(numeric.lt(threshold).sum())
        if positive_rows < minimum_class_rows or negative_rows < minimum_class_rows:
            continue
        prevalence = positive_rows / len(numeric)
        candidates.append(
            (
                abs(prevalence - desired_positive_fraction),
                -float(threshold),
                prevalence,
            )
        )

    if not candidates:
        raise ValueError("No training-only threshold preserved both target classes.")

    candidates.sort()
    _, negative_threshold, prevalence = candidates[0]
    return -negative_threshold, prevalence



def build_nearest_flow_target_table(
    edges_path: Path,
    flow_labels_path: Path,
    edge_chunk_rows: int = 500_000,
    label_chunk_rows: int = 500_000,
) -> pd.DataFrame:
    """Build one deterministic nearest-flow target row per event without loading all edges at once."""
    required_edge_columns = {
        ID_COLUMN,
        "flow_id",
        "absolute_flow_start_lag_s",
        "overlap_seconds",
        "source_row_number",
    }
    edge_header = pd.read_csv(edges_path, nrows=0)
    missing_edges = sorted(required_edge_columns - set(edge_header.columns))
    if missing_edges:
        raise KeyError(f"Primary edge table is missing required columns: {missing_edges}")

    chunk_winners: list[pd.DataFrame] = []
    edge_usecols = sorted(required_edge_columns)
    for chunk in pd.read_csv(
        edges_path,
        usecols=edge_usecols,
        chunksize=edge_chunk_rows,
        low_memory=False,
    ):
        chunk[ID_COLUMN] = chunk[ID_COLUMN].astype("string")
        chunk["flow_id"] = chunk["flow_id"].astype("string")
        chunk["absolute_flow_start_lag_s"] = pd.to_numeric(
            chunk["absolute_flow_start_lag_s"], errors="coerce"
        )
        chunk["overlap_seconds"] = pd.to_numeric(
            chunk["overlap_seconds"], errors="coerce"
        )
        chunk["source_row_number"] = pd.to_numeric(
            chunk["source_row_number"], errors="coerce"
        )
        chunk = chunk.dropna(
            subset=[ID_COLUMN, "flow_id", "absolute_flow_start_lag_s"]
        )
        if chunk.empty:
            continue
        chunk["_overlap_sort"] = chunk["overlap_seconds"].fillna(-1.0)
        chunk["_source_row_sort"] = chunk["source_row_number"].fillna(np.inf)
        chunk = chunk.sort_values(
            [
                ID_COLUMN,
                "absolute_flow_start_lag_s",
                "_overlap_sort",
                "_source_row_sort",
                "flow_id",
            ],
            ascending=[True, True, False, True, True],
            kind="mergesort",
        )
        chunk_winners.append(chunk.drop_duplicates(ID_COLUMN, keep="first"))

    if not chunk_winners:
        return pd.DataFrame(
            columns=[
                ID_COLUMN,
                "nearest_flow_id",
                "nearest_flow_binary_label",
                "nearest_flow_attack_category",
                "nearest_flow_absolute_start_lag_s",
                "nearest_flow_overlap_seconds",
                "nearest_flow_source_row_number",
            ]
        )

    nearest = pd.concat(chunk_winners, ignore_index=True)
    nearest = nearest.sort_values(
        [
            ID_COLUMN,
            "absolute_flow_start_lag_s",
            "_overlap_sort",
            "_source_row_sort",
            "flow_id",
        ],
        ascending=[True, True, False, True, True],
        kind="mergesort",
    ).drop_duplicates(ID_COLUMN, keep="first")

    label_header = pd.read_csv(flow_labels_path, nrows=0)
    required_label_columns = {
        "flow_id",
        "nf3_binary_label",
        "nf3_attack_category",
    }
    missing_labels = sorted(required_label_columns - set(label_header.columns))
    if missing_labels:
        raise KeyError(f"Flow-label table is missing required columns: {missing_labels}")

    needed_flow_ids = set(nearest["flow_id"].dropna().astype(str).tolist())
    label_parts: list[pd.DataFrame] = []
    for chunk in pd.read_csv(
        flow_labels_path,
        usecols=sorted(required_label_columns),
        chunksize=label_chunk_rows,
        low_memory=False,
    ):
        chunk["flow_id"] = chunk["flow_id"].astype("string")
        selected = chunk.loc[chunk["flow_id"].isin(needed_flow_ids)].copy()
        if not selected.empty:
            label_parts.append(selected)

    if not label_parts:
        raise ValueError("No nearest-flow identifiers were found in nf3_flow_labels_used_v2.csv.gz.")

    labels = pd.concat(label_parts, ignore_index=True).drop_duplicates("flow_id")
    nearest = nearest.merge(labels, on="flow_id", how="left", validate="many_to_one")
    nearest["nf3_binary_label"] = pd.to_numeric(
        nearest["nf3_binary_label"], errors="coerce"
    )

    nearest = nearest.rename(
        columns={
            "flow_id": "nearest_flow_id",
            "nf3_binary_label": "nearest_flow_binary_label",
            "nf3_attack_category": "nearest_flow_attack_category",
            "absolute_flow_start_lag_s": "nearest_flow_absolute_start_lag_s",
            "overlap_seconds": "nearest_flow_overlap_seconds",
            "source_row_number": "nearest_flow_source_row_number",
        }
    )
    return nearest[
        [
            ID_COLUMN,
            "nearest_flow_id",
            "nearest_flow_binary_label",
            "nearest_flow_attack_category",
            "nearest_flow_absolute_start_lag_s",
            "nearest_flow_overlap_seconds",
            "nearest_flow_source_row_number",
        ]
    ]

def construct_model_target(
    data: pd.DataFrame,
    chronological_assignment: pd.Series,
) -> tuple[pd.Series, dict[str, Any]]:
    """Select a viable binary target using a fixed, documented fallback order."""
    raw_target = pd.to_numeric(data[RAW_TARGET_COLUMN], errors="raise").astype("int8")
    train_mask = chronological_assignment.eq("train")

    if has_both_classes(raw_target) and has_both_classes(raw_target.loc[train_mask]):
        definition = {
            "mode": "raw_any_attack_window_label",
            "model_target_column": TARGET_COLUMN,
            "source_column": RAW_TARGET_COLUMN,
            "operator": "identity",
            "threshold": None,
            "training_positive_fraction": require_float(
                raw_target.loc[train_mask].mean(),
                "raw training target prevalence",
            ),
            "full_positive_fraction": require_float(
                raw_target.mean(),
                "raw full target prevalence",
            ),
            "reason": "The raw Notebook 03 target contained both classes overall and in chronological training.",
        }
        return raw_target, definition

    failures: dict[str, str] = {}

    # Preferred fallback: the preserved label of the primary-window flow whose
    # start time is nearest to the aligned event time. The selection uses only
    # temporal/relation fields, never the label value.
    nearest_column = "nearest_flow_binary_label"
    if nearest_column in data.columns:
        nearest_numeric = pd.to_numeric(data[nearest_column], errors="coerce")
        missing_rows = require_int(nearest_numeric.isna().sum(), "missing nearest-flow labels")
        invalid_values = sorted(
            set(nearest_numeric.dropna().unique().tolist()) - {0, 1, 0.0, 1.0}
        )
        if missing_rows:
            failures[nearest_column] = f"contains {missing_rows:,} missing rows"
        elif invalid_values:
            failures[nearest_column] = f"contains non-binary values: {invalid_values}"
        else:
            nearest_target = nearest_numeric.astype("int8")
            if not has_both_classes(nearest_target):
                failures[nearest_column] = "nearest-flow labels are single-class overall"
            elif not has_both_classes(nearest_target.loc[train_mask]):
                failures[nearest_column] = "nearest-flow labels are single-class in chronological training"
            else:
                definition = {
                    "mode": "nearest_flow_binary_label",
                    "model_target_column": TARGET_COLUMN,
                    "source_column": nearest_column,
                    "operator": "identity",
                    "threshold": None,
                    "selection_rule": (
                        "Minimum absolute flow-start lag; ties resolved by greatest overlap, "
                        "smallest source row number, then flow_id."
                    ),
                    "training_positive_fraction": require_float(
                        nearest_target.loc[train_mask].mean(),
                        "nearest-flow training target prevalence",
                    ),
                    "full_positive_fraction": require_float(
                        nearest_target.mean(),
                        "nearest-flow full target prevalence",
                    ),
                    "raw_target_unique_values": sorted(
                        raw_target.unique().astype(int).tolist()
                    ),
                    "raw_target_positive_fraction": require_float(
                        raw_target.mean(),
                        "raw target prevalence",
                    ),
                    "reason": (
                        "The any-attack window label was saturated. A deterministic nearest-flow "
                        "label preserves a binary NF3 target without using labels to choose the flow."
                    ),
                }
                return nearest_target, definition
    else:
        failures[nearest_column] = "column_missing"

    # Secondary fallback retained for compatibility when the target-only burden
    # measures genuinely vary in chronological training.
    source_candidates = ["nf3_attack_flow_fraction", "nf3_attack_flow_count"]
    for source_column in source_candidates:
        if source_column not in data.columns:
            failures[source_column] = "column_missing"
            continue

        source_values = pd.to_numeric(data[source_column], errors="coerce")
        if source_values.isna().any():
            failures[source_column] = "contains_missing_values"
            continue

        try:
            threshold, training_prevalence = choose_training_burden_threshold(
                source_values.loc[train_mask],
                desired_positive_fraction=HIGH_BURDEN_DESIRED_POSITIVE_FRACTION,
                minimum_class_rows=HIGH_BURDEN_MIN_CLASS_ROWS,
            )
        except ValueError as exc:
            failures[source_column] = str(exc)
            continue

        derived = source_values.ge(threshold).astype("int8")
        if not has_both_classes(derived.loc[train_mask]):
            failures[source_column] = "derived training target remained single-class"
            continue

        definition = {
            "mode": "derived_high_nf3_attack_burden",
            "model_target_column": TARGET_COLUMN,
            "source_column": source_column,
            "operator": ">=",
            "threshold": threshold,
            "desired_training_positive_fraction": HIGH_BURDEN_DESIRED_POSITIVE_FRACTION,
            "training_positive_fraction": training_prevalence,
            "full_positive_fraction": require_float(
                derived.mean(),
                "derived full target prevalence",
            ),
            "raw_target_unique_values": sorted(raw_target.unique().astype(int).tolist()),
            "raw_target_positive_fraction": require_float(
                raw_target.mean(),
                "raw target prevalence",
            ),
            "reason": (
                "The raw and nearest-flow targets were not viable. The burden threshold "
                "was selected from chronological training values only."
            ),
            "failed_source_candidates": failures,
        }
        return derived, definition

    raise ValueError(
        "No valid binary target could be constructed. The any-attack window target is "
        "single-class, and the deterministic nearest-flow label or burden fallbacks were "
        f"not viable. Candidate diagnostics: {failures}"
    )

def safe_average_precision(y_true: pd.Series | np.ndarray, probabilities: np.ndarray) -> float:
    if not has_both_classes(y_true):
        return float("nan")
    return float(average_precision_score(y_true, probabilities))


def safe_roc_auc(y_true: pd.Series | np.ndarray, probabilities: np.ndarray) -> float:
    if not has_both_classes(y_true):
        return float("nan")
    return float(roc_auc_score(y_true, probabilities))


def select_f1_threshold(y_true: pd.Series, probabilities: np.ndarray) -> dict[str, float | str]:
    """Select validation F1 threshold; use 0.5 when validation is single-class."""
    if not has_both_classes(y_true):
        threshold = 0.5
        predicted = (probabilities >= threshold).astype(int)
        return {
            "threshold": threshold,
            "policy": "fixed_0.5_validation_single_class",
            "validation_f1": float(f1_score(y_true, predicted, zero_division=0)),
            "validation_precision": float(precision_score(y_true, predicted, zero_division=0)),
            "validation_recall": float(recall_score(y_true, predicted, zero_division=0)),
        }

    precision_values, recall_values, thresholds = precision_recall_curve(y_true, probabilities)
    if thresholds.size == 0:
        threshold = 0.5
        predicted = (probabilities >= threshold).astype(int)
        return {
            "threshold": threshold,
            "policy": "fixed_0.5_no_validation_thresholds",
            "validation_f1": float(f1_score(y_true, predicted, zero_division=0)),
            "validation_precision": float(precision_score(y_true, predicted, zero_division=0)),
            "validation_recall": float(recall_score(y_true, predicted, zero_division=0)),
        }

    precision_at_threshold = precision_values[:-1]
    recall_at_threshold = recall_values[:-1]
    denominator = precision_at_threshold + recall_at_threshold
    f1_values = np.divide(
        2.0 * precision_at_threshold * recall_at_threshold,
        denominator,
        out=np.zeros_like(denominator, dtype=float),
        where=denominator > 0,
    )

    best_f1 = float(np.nanmax(f1_values))
    candidates = np.flatnonzero(np.isclose(f1_values, best_f1, rtol=1e-12, atol=1e-12))
    if candidates.size > 1:
        candidate_recalls = recall_at_threshold[candidates]
        candidates = candidates[np.isclose(candidate_recalls, np.nanmax(candidate_recalls))]
    best_index = int(candidates[np.argmin(thresholds[candidates])])

    return {
        "threshold": float(thresholds[best_index]),
        "policy": "maximize_validation_f1_tie_higher_recall_then_lower_threshold",
        "validation_f1": float(f1_values[best_index]),
        "validation_precision": float(precision_at_threshold[best_index]),
        "validation_recall": float(recall_at_threshold[best_index]),
    }


def classification_metrics(
    y_true: pd.Series,
    probabilities: np.ndarray,
    threshold: float,
) -> dict[str, float | int]:
    predicted = (probabilities >= threshold).astype(int)
    matrix = confusion_matrix(y_true, predicted, labels=[0, 1])
    tn, fp, fn, tp = [int(value) for value in matrix.ravel().tolist()]
    both_classes = has_both_classes(y_true)

    return {
        "auprc": safe_average_precision(y_true, probabilities),
        "roc_auc": safe_roc_auc(y_true, probabilities),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "balanced_accuracy": (
            float(balanced_accuracy_score(y_true, predicted))
            if both_classes
            else float("nan")
        ),
        "brier_score": float(brier_score_loss(y_true, probabilities)),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "positive_rate": float(np.mean(np.asarray(y_true, dtype=float))),
        "both_classes_present": int(both_classes),
    }


def stratified_sample_positions(y: pd.Series, maximum: int, random_state: int) -> np.ndarray:
    """Return position indices with approximate class preservation."""
    if len(y) <= maximum:
        return np.arange(len(y), dtype=int)

    rng = np.random.default_rng(random_state)
    positions: list[int] = []
    y_array = np.asarray(y, dtype=int)
    for label in [0, 1]:
        label_positions = np.flatnonzero(y_array == label)
        if label_positions.size == 0:
            continue
        allocation = max(1, int(round(maximum * len(label_positions) / len(y_array))))
        allocation = min(allocation, len(label_positions))
        positions.extend(rng.choice(label_positions, size=allocation, replace=False).tolist())

    if len(positions) > maximum:
        positions = rng.choice(np.asarray(positions, dtype=int), size=maximum, replace=False).tolist()
    return np.asarray(sorted(set(positions)), dtype=int)


def measure_single_event_latency_ms(
    fitted_pipeline: Pipeline,
    X_test: pd.DataFrame,
    maximum_events: int,
    random_state: int,
) -> dict[str, float | int]:
    """Measure end-to-end single-event predict_proba latency."""
    if X_test.empty:
        return {
            "latency_events": 0,
            "latency_mean_ms": float("nan"),
            "latency_p50_ms": float("nan"),
            "latency_p95_ms": float("nan"),
            "latency_max_ms": float("nan"),
        }

    rng = np.random.default_rng(random_state)
    count = min(maximum_events, len(X_test))
    selected_positions = rng.choice(len(X_test), size=count, replace=False)
    single_rows = [X_test.iloc[[int(position)]] for position in selected_positions]

    for row in single_rows[: min(10, len(single_rows))]:
        fitted_pipeline.predict_proba(row)

    elapsed_ms: list[float] = []
    for row in single_rows:
        start = time.perf_counter_ns()
        fitted_pipeline.predict_proba(row)
        stop = time.perf_counter_ns()
        elapsed_ms.append((stop - start) / 1_000_000.0)

    values = np.asarray(elapsed_ms, dtype=float)
    return {
        "latency_events": int(values.size),
        "latency_mean_ms": float(values.mean()),
        "latency_p50_ms": float(np.percentile(values, 50)),
        "latency_p95_ms": float(np.percentile(values, 95)),
        "latency_max_ms": float(values.max()),
    }


## 2. Load Notebook 03 outputs and merge the primary target

In [ ]:
for required_path in [FEATURES_IN, TARGETS_IN]:
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Required Notebook 03 output was not found: {required_path}\n"
            "Run Notebook 03 successfully from the project root before running Notebook 04."
        )

features = pd.read_csv(FEATURES_IN, low_memory=False)
targets_all = pd.read_csv(TARGETS_IN, low_memory=False)

required_feature_columns = {ID_COLUMN, TIME_COLUMN, GROUP_COLUMN}
missing_feature_columns = sorted(required_feature_columns - set(features.columns))
if missing_feature_columns:
    raise KeyError(f"Feature bank is missing required columns: {missing_feature_columns}")

required_target_columns = {
    ID_COLUMN,
    "window_name",
    RAW_TARGET_COLUMN,
    "nf3_attack_flow_fraction",
    "nf3_attack_flow_count",
}
missing_target_columns = sorted(required_target_columns - set(targets_all.columns))
if missing_target_columns:
    raise KeyError(f"Target table is missing required columns: {missing_target_columns}")

target_only_columns = [
    ID_COLUMN,
    RAW_TARGET_COLUMN,
    "nf3_binary_label_available_flag",
    "nf3_attack_category_available_flag",
    "nf3_attack_flow_count",
    "nf3_benign_flow_count",
    "nf3_unknown_binary_label_count",
    "nf3_attack_flow_fraction",
    "nf3_distinct_attack_category_count",
    "nf3_attack_categories_present",
    "nf3_attack_category_counts_json",
]
target_only_columns = [column for column in target_only_columns if column in targets_all.columns]

primary_targets = targets_all.loc[
    targets_all["window_name"].astype("string").eq(PRIMARY_WINDOW_NAME),
    target_only_columns,
].copy()

if primary_targets.empty:
    raise ValueError(f"No target rows were found for window_name={PRIMARY_WINDOW_NAME!r}.")
if primary_targets[ID_COLUMN].duplicated().any():
    raise ValueError("Primary target table contains duplicate event_id values.")
if features[ID_COLUMN].duplicated().any():
    raise ValueError("Primary feature bank contains duplicate event_id values.")

model_data = features.merge(
    primary_targets,
    on=ID_COLUMN,
    how="inner",
    validate="one_to_one",
)

nearest_targets = pd.DataFrame()
if EDGES_IN.is_file() and FLOW_LABELS_IN.is_file():
    nearest_targets = build_nearest_flow_target_table(EDGES_IN, FLOW_LABELS_IN)
    if nearest_targets[ID_COLUMN].duplicated().any():
        raise ValueError("Nearest-flow target table contains duplicate event_id values.")
    model_data = model_data.merge(
        nearest_targets,
        on=ID_COLUMN,
        how="left",
        validate="one_to_one",
    )
else:
    missing_optional = [
        str(path)
        for path in [EDGES_IN, FLOW_LABELS_IN]
        if not path.is_file()
    ]
    warnings.warn(
        "Nearest-flow fallback is unavailable because these Notebook 03 artifacts "
        f"were not found: {missing_optional}"
    )

if len(model_data) != len(features):
    raise ValueError(
        f"Target merge changed the event count: features={len(features):,}, merged={len(model_data):,}."
    )

model_data[TIME_COLUMN] = pd.to_datetime(model_data[TIME_COLUMN], utc=True, errors="raise")
model_data[ID_COLUMN] = model_data[ID_COLUMN].astype("string")
model_data[GROUP_COLUMN] = model_data[GROUP_COLUMN].astype("string")
model_data[RAW_TARGET_COLUMN] = pd.to_numeric(
    model_data[RAW_TARGET_COLUMN], errors="raise"
).astype("int8")

if model_data[[ID_COLUMN, GROUP_COLUMN, TIME_COLUMN, RAW_TARGET_COLUMN]].isna().any().any():
    raise ValueError("Required identifier, time, group, or raw-target fields contain missing values.")
if not set(model_data[RAW_TARGET_COLUMN].unique().tolist()).issubset({0, 1}):
    raise ValueError(f"{RAW_TARGET_COLUMN} must contain only 0 and 1.")

model_data = model_data.sort_values([TIME_COLUMN, ID_COLUMN], kind="mergesort").reset_index(drop=True)
provisional_chronological_split = time_only_chronological_assignments(model_data)
model_target, target_definition = construct_model_target(
    model_data,
    provisional_chronological_split,
)
model_data[TARGET_COLUMN] = model_target.astype("int8")

if not has_both_classes(model_data.loc[provisional_chronological_split.eq("train"), TARGET_COLUMN]):
    raise ValueError("The final chronological training target does not contain both classes.")

TARGET_DEFINITION_OUT.write_text(
    json.dumps(target_definition, indent=2, default=str),
    encoding="utf-8",
)

alignment_summary: dict[str, Any] = {}
if ALIGNMENT_SUMMARY_IN.is_file():
    alignment_summary = json.loads(ALIGNMENT_SUMMARY_IN.read_text(encoding="utf-8"))

print(f"Point events:             {len(model_data):,}")
print(f"Unique aircraft keys:    {model_data[GROUP_COLUMN].nunique():,}")
print(f"Raw target values:       {sorted(model_data[RAW_TARGET_COLUMN].unique().tolist())}")
if "nearest_flow_binary_label" in model_data.columns:
    nearest_distribution = (
        pd.to_numeric(model_data["nearest_flow_binary_label"], errors="coerce")
        .value_counts(dropna=False)
        .sort_index()
        .to_dict()
    )
    print(f"Nearest-flow labels:     {nearest_distribution}")
print(f"Model target mode:       {target_definition['mode']}")
print(f"Model target source:     {target_definition['source_column']}")
if target_definition.get("threshold") is not None:
    print(f"Model target threshold:  {target_definition['operator']} {target_definition['threshold']}")
print(f"Positive model targets:  {require_int(model_data[TARGET_COLUMN].sum(), 'positive target count'):,}")
print(f"Negative model targets:  {require_int(model_data[TARGET_COLUMN].eq(0).sum(), 'negative target count'):,}")
print(f"Model prevalence:        {require_float(model_data[TARGET_COLUMN].mean(), 'model target prevalence'):.4f}")
print(f"Event range:             {model_data[TIME_COLUMN].min()} to {model_data[TIME_COLUMN].max()}")

if alignment_summary:
    selected_mode = alignment_summary.get("alignment", {}).get("selected_mode")
    controlled = alignment_summary.get("interpretation", {}).get("controlled_mapping_active")
    print(f"Alignment mode:          {selected_mode}")
    print(f"Controlled mapping:      {controlled}")


## 3. Define leakage-safe physical, cyber, and fused feature sets

In [ ]:
TARGET_DERIVED_COLUMNS = {
    TARGET_COLUMN,
    RAW_TARGET_COLUMN,
    "nf3_binary_label_available_flag",
    "nf3_attack_category_available_flag",
    "nf3_attack_flow_count",
    "nf3_benign_flow_count",
    "nf3_unknown_binary_label_count",
    "nf3_attack_flow_fraction",
    "nf3_distinct_attack_category_count",
    "nf3_attack_categories_present",
    "nf3_attack_category_counts_json",
    "nf3_binary_label_raw",
    "nf3_attack_category_raw",
    "nf3_binary_label",
    "nf3_attack_category",
    "Label",
    "Attack",
    "nearest_flow_binary_label",
    "nearest_flow_attack_category",
    "nearest_flow_id",
    "nearest_flow_absolute_start_lag_s",
    "nearest_flow_overlap_seconds",
    "nearest_flow_source_row_number",
}

IDENTIFIER_AND_PROVENANCE_COLUMNS = {
    ID_COLUMN,
    GROUP_COLUMN,
    "flight_track_id",
    "hex",
    "callsign",
    "fr24_id",
    "episode_id",
    "source_filename",
    "source_line_number",
    "source_aircraft_index",
    "snapshot_time_utc",
    TIME_COLUMN,
    "episode_start_time_utc",
    "episode_end_time_utc",
    "aligned_event_time_utc",
    "window_start_utc",
    "window_end_utc",
}

# Audit-only alignment fields are excluded so a model cannot exploit the mechanics
# of controlled replay instead of learning physical or cyber behavior.
ALIGNMENT_AUDIT_ONLY_COLUMNS = {
    "alignment_version",
    "alignment_mode",
    "alignment_is_controlled",
    "alignment_time_shift_s",
    "replay_cycle_index",
    "window_name",
    "window_half_width_s",
}

CALENDAR_COLUMNS = {
    "event_year_utc",
    "event_month_utc",
    "event_day_utc",
    "event_hour_utc",
    "event_day_of_week_utc",
    "event_weekend_flag",
}

PHYSICAL_NUMERIC_CANDIDATES = [
    "event_lag_seconds",
    "stale_event_flag",
    "latitude",
    "longitude",
    "distance_miles",
    "altitude_ft",
    "ground_speed_kt",
    "vertical_speed_fpm",
    "track_degrees",
    "missing_hex_flag",
    "missing_callsign_flag",
    "gap_from_previous_same_aircraft_s",
    "episode_sequence_number",
    "episode_size",
    "episode_max_internal_gap_s",
    "episode_gap_threshold_s",
    "episode_position",
    "multi_observation_episode_flag",
    "duration_observable_flag",
    "observed_duration_s",
    "prior_event_count",
    "events_previous_1h",
    "events_previous_24h",
    "events_previous_7d",
    "events_previous_30d",
    "time_since_previous_event_s",
    "prior_mean_altitude_ft",
    "altitude_ft_deviation_from_prior_mean",
    "prior_mean_ground_speed_kt",
    "ground_speed_kt_deviation_from_prior_mean",
    "prior_mean_distance_miles",
    "distance_miles_deviation_from_prior_mean",
    "prior_mean_vertical_speed_fpm",
    "vertical_speed_fpm_deviation_from_prior_mean",
    "recurrence_1h_flag",
    "recurrence_24h_flag",
    "recurrence_7d_flag",
    "recurrence_30d_flag",
    "first_observed_aircraft_event_flag",
    "site_events_previous_1h",
    "site_events_previous_24h",
    "site_events_previous_7d",
]

PHYSICAL_CATEGORICAL_CANDIDATES = [
    "source_type",
    "aircraft_key_source",
    "deduplication_status",
]

if INCLUDE_CALENDAR_FEATURES:
    PHYSICAL_NUMERIC_CANDIDATES.extend(sorted(CALENDAR_COLUMNS))

# Add circular representations without retaining target information.
if "track_degrees" in model_data.columns:
    track_radians = np.deg2rad(pd.to_numeric(model_data["track_degrees"], errors="coerce"))
    model_data["track_sin"] = np.sin(track_radians)
    model_data["track_cos"] = np.cos(track_radians)
    PHYSICAL_NUMERIC_CANDIDATES.extend(["track_sin", "track_cos"])

if INCLUDE_CALENDAR_FEATURES and "event_hour_utc" in model_data.columns:
    hour_radians = 2.0 * np.pi * pd.to_numeric(model_data["event_hour_utc"], errors="coerce") / 24.0
    model_data["event_hour_sin"] = np.sin(hour_radians)
    model_data["event_hour_cos"] = np.cos(hour_radians)
    PHYSICAL_NUMERIC_CANDIDATES.extend(["event_hour_sin", "event_hour_cos"])

CYBER_NUMERIC_CANDIDATES = [
    column
    for column in model_data.columns
    if column.startswith("nf3_") and column not in TARGET_DERIVED_COLUMNS
]

# Window truncation is a non-label cyber measurement-quality field.
if "window_boundary_truncated_flag" in model_data.columns:
    CYBER_NUMERIC_CANDIDATES.append("window_boundary_truncated_flag")


def existing_unique(columns: Iterable[str]) -> list[str]:
    seen: set[str] = set()
    output: list[str] = []
    for column in columns:
        if column in model_data.columns and column not in seen:
            output.append(column)
            seen.add(column)
    return output


physical_numeric = existing_unique(PHYSICAL_NUMERIC_CANDIDATES)
physical_categorical = existing_unique(PHYSICAL_CATEGORICAL_CANDIDATES)
cyber_numeric = existing_unique(CYBER_NUMERIC_CANDIDATES)
cyber_categorical: list[str] = []

for column in physical_numeric + cyber_numeric:
    model_data[column] = pd.to_numeric(model_data[column], errors="coerce")
for column in physical_categorical + cyber_categorical:
    model_data[column] = model_data[column].astype("string")

feature_sets = {
    "physical": {
        "numeric": physical_numeric,
        "categorical": physical_categorical,
    },
    "cyber": {
        "numeric": cyber_numeric,
        "categorical": cyber_categorical,
    },
    "fused": {
        "numeric": existing_unique(physical_numeric + cyber_numeric),
        "categorical": existing_unique(physical_categorical + cyber_categorical),
    },
}

all_predictors = set()
for specification in feature_sets.values():
    all_predictors.update(specification["numeric"])
    all_predictors.update(specification["categorical"])

leakage_intersection = sorted(all_predictors & TARGET_DERIVED_COLUMNS)
identifier_intersection = sorted(all_predictors & IDENTIFIER_AND_PROVENANCE_COLUMNS)
alignment_audit_intersection = sorted(all_predictors & ALIGNMENT_AUDIT_ONLY_COLUMNS)

if leakage_intersection:
    raise AssertionError(f"Target leakage columns entered the predictor bank: {leakage_intersection}")
if identifier_intersection:
    raise AssertionError(f"Identifier/provenance columns entered the predictor bank: {identifier_intersection}")
if alignment_audit_intersection:
    raise AssertionError(f"Alignment audit fields entered the predictor bank: {alignment_audit_intersection}")

feature_manifest = {
    "model_version": MODEL_VERSION,
    "target": TARGET_COLUMN,
    "primary_window": PRIMARY_WINDOW_NAME,
    "calendar_features_included": INCLUDE_CALENDAR_FEATURES,
    "feature_sets": feature_sets,
    "excluded_target_derived_columns": sorted(TARGET_DERIVED_COLUMNS),
    "excluded_identifier_and_provenance_columns": sorted(IDENTIFIER_AND_PROVENANCE_COLUMNS),
    "excluded_alignment_audit_columns": sorted(ALIGNMENT_AUDIT_ONLY_COLUMNS),
}
FEATURE_MANIFEST_OUT.write_text(json.dumps(feature_manifest, indent=2), encoding="utf-8")

feature_overview = pd.DataFrame(
    [
        {
            "feature_set": name,
            "numeric_candidates": len(specification["numeric"]),
            "categorical_candidates": len(specification["categorical"]),
            "total_candidates": len(specification["numeric"]) + len(specification["categorical"]),
        }
        for name, specification in feature_sets.items()
    ]
)
display(feature_overview)

if any(row["total_candidates"] == 0 for row in feature_overview.to_dict("records")):
    raise ValueError("At least one feature set contains no predictors.")

## 4. Create fixed chronological and aircraft-group sensitivity splits

In [ ]:
def grouped_split_assignments(
    data: pd.DataFrame,
    maximum_attempts: int = 200,
) -> tuple[pd.Series | None, int | None]:
    """Try to create aircraft-disjoint partitions with both classes in each split."""
    groups = data[GROUP_COLUMN].astype("string")
    target = data[TARGET_COLUMN]

    for attempt in range(maximum_attempts):
        seed = RANDOM_STATE + attempt
        first_splitter = GroupShuffleSplit(
            n_splits=1,
            train_size=TRAIN_FRACTION,
            random_state=seed,
        )
        train_positions, temporary_positions = next(
            first_splitter.split(data, target, groups=groups)
        )

        temporary = data.iloc[temporary_positions]
        temporary_groups = groups.iloc[temporary_positions]
        second_splitter = GroupShuffleSplit(
            n_splits=1,
            train_size=0.5,
            random_state=seed + 10_000,
        )
        validation_relative, test_relative = next(
            second_splitter.split(
                temporary,
                temporary[TARGET_COLUMN],
                groups=temporary_groups,
            )
        )

        validation_positions = temporary_positions[validation_relative]
        test_positions = temporary_positions[test_relative]

        assignment = pd.Series("", index=data.index, dtype="string")
        assignment.iloc[train_positions] = "train"
        assignment.iloc[validation_positions] = "validation"
        assignment.iloc[test_positions] = "test"

        if all(
            has_both_classes(data.loc[assignment.eq(split_name), TARGET_COLUMN])
            for split_name in ["train", "validation", "test"]
        ):
            return assignment, seed

    warnings.warn(
        "Aircraft-group sensitivity split was skipped because both target classes could not be "
        f"placed in every partition after {maximum_attempts} attempts.",
        RuntimeWarning,
    )
    return None, None


split_assignments = model_data[
    [ID_COLUMN, GROUP_COLUMN, TIME_COLUMN, RAW_TARGET_COLUMN, TARGET_COLUMN]
].copy()
split_assignments["chronological_split"] = time_only_chronological_assignments(model_data)

# Training must have both classes. Validation/test single-class partitions are
# allowed, but class-dependent metrics are reported as NaN and thresholding uses 0.5.
for split_name in ["train", "validation", "test"]:
    values = model_data.loc[
        split_assignments["chronological_split"].eq(split_name),
        TARGET_COLUMN,
    ]
    if split_name == "train" and not has_both_classes(values):
        raise ValueError("Chronological training does not contain both model-target classes.")
    if split_name != "train" and not has_both_classes(values):
        warnings.warn(
            f"Chronological {split_name} is single-class; AUPRC, ROC-AUC, and balanced accuracy "
            "will be reported as NaN for that partition.",
            RuntimeWarning,
        )

grouped_seed: int | None = None
grouped_sensitivity_available = False
if RUN_GROUPED_SENSITIVITY:
    grouped_assignment, grouped_seed = grouped_split_assignments(model_data)
    if grouped_assignment is not None:
        split_assignments["aircraft_group_split"] = grouped_assignment
        grouped_sensitivity_available = True
    else:
        split_assignments["aircraft_group_split"] = pd.Series(
            pd.NA, index=model_data.index, dtype="string"
        )
else:
    split_assignments["aircraft_group_split"] = pd.Series(
        pd.NA, index=model_data.index, dtype="string"
    )

if grouped_sensitivity_available:
    for left_name, right_name in [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test"),
    ]:
        left_groups = set(
            split_assignments.loc[
                split_assignments["aircraft_group_split"].eq(left_name),
                GROUP_COLUMN,
            ].astype(str)
        )
        right_groups = set(
            split_assignments.loc[
                split_assignments["aircraft_group_split"].eq(right_name),
                GROUP_COLUMN,
            ].astype(str)
        )
        overlap = left_groups & right_groups
        if overlap:
            raise AssertionError(
                f"Grouped split leakage: {len(overlap)} aircraft keys occur in both "
                f"{left_name} and {right_name}."
            )

split_assignments.to_csv(SPLITS_OUT, index=False)

split_summary_rows: list[dict[str, Any]] = []
for strategy_column in ["chronological_split", "aircraft_group_split"]:
    if split_assignments[strategy_column].isna().all():
        continue
    for split_name in ["train", "validation", "test"]:
        mask = split_assignments[strategy_column].eq(split_name)
        split_targets = split_assignments.loc[mask, TARGET_COLUMN]
        split_summary_rows.append(
            {
                "split_strategy": strategy_column,
                "split": split_name,
                "rows": require_int(mask.sum(), f"{strategy_column} {split_name} rows"),
                "aircraft_keys": require_int(
                    split_assignments.loc[mask, GROUP_COLUMN].nunique(),
                    f"{strategy_column} {split_name} aircraft count",
                ),
                "positive_targets": require_int(
                    split_targets.sum(),
                    f"{strategy_column} {split_name} positive targets",
                ),
                "both_classes_present": int(has_both_classes(split_targets)),
                "target_prevalence": require_float(
                    split_targets.mean(),
                    f"{strategy_column} {split_name} target prevalence",
                ),
                "start_time_utc": split_assignments.loc[mask, TIME_COLUMN].min(),
                "end_time_utc": split_assignments.loc[mask, TIME_COLUMN].max(),
            }
        )

split_summary = pd.DataFrame(split_summary_rows)
display(split_summary)


## 5. Model and preprocessing definitions

In [ ]:
@dataclass(frozen=True)
class ModelSpecification:
    name: str
    estimator: Any
    scale_numeric: bool


model_specifications = [
    ModelSpecification(
        name="dummy_prior",
        estimator=DummyClassifier(strategy="prior", random_state=RANDOM_STATE),
        scale_numeric=False,
    ),
    ModelSpecification(
        name="logistic_regression",
        estimator=LogisticRegression(
            class_weight="balanced",
            max_iter=2_000,
            solver="liblinear",
            random_state=RANDOM_STATE,
        ),
        scale_numeric=True,
    ),
    ModelSpecification(
        name="decision_tree",
        estimator=DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        scale_numeric=False,
    ),
    ModelSpecification(
        name="random_forest",
        estimator=RandomForestClassifier(
            n_estimators=RANDOM_FOREST_TREES,
            max_depth=12,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        scale_numeric=False,
    ),
]


def run_enabled(
    split_strategy: str,
    feature_set_name: str,
    model_name: str,
) -> bool:
    if split_strategy == "chronological_primary":
        return True
    if split_strategy == "aircraft_group_sensitivity":
        return (
            feature_set_name in GROUPED_SENSITIVITY_FEATURE_SETS
            and model_name in GROUPED_SENSITIVITY_MODELS
        )
    return False


def usable_columns_for_training(
    training_data: pd.DataFrame,
    numeric_candidates: list[str],
    categorical_candidates: list[str],
) -> tuple[list[str], list[str], list[str]]:
    usable_numeric: list[str] = []
    usable_categorical: list[str] = []
    dropped: list[str] = []

    for column in numeric_candidates:
        non_missing = pd.to_numeric(training_data[column], errors="coerce").dropna()
        if non_missing.empty or non_missing.nunique() <= 1:
            dropped.append(column)
        else:
            usable_numeric.append(column)

    for column in categorical_candidates:
        non_missing = training_data[column].astype("string").dropna()
        if non_missing.empty or non_missing.nunique() <= 1:
            dropped.append(column)
        else:
            usable_categorical.append(column)

    return usable_numeric, usable_categorical, dropped


def build_pipeline(
    numeric_columns: list[str],
    categorical_columns: list[str],
    specification: ModelSpecification,
) -> Pipeline:
    transformers: list[tuple[str, Pipeline, list[str]]] = []

    if numeric_columns:
        numeric_steps: list[tuple[str, Any]] = [
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ]
        if specification.scale_numeric:
            numeric_steps.append(("scaler", StandardScaler(with_mean=False)))
        transformers.append(("numeric", Pipeline(numeric_steps), numeric_columns))

    if categorical_columns:
        categorical_pipeline = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("one_hot", make_one_hot_encoder()),
            ]
        )
        transformers.append(("categorical", categorical_pipeline, categorical_columns))

    if not transformers:
        raise ValueError("No usable predictor columns remained after training-only screening.")

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
        verbose_feature_names_out=True,
    )

    return Pipeline(
        [
            ("preprocess", preprocessor),
            ("model", clone(specification.estimator)),
        ]
    )


def transformed_feature_names(fitted_pipeline: Pipeline) -> np.ndarray:
    preprocessor = fitted_pipeline.named_steps["preprocess"]
    return np.asarray(preprocessor.get_feature_names_out(), dtype=str)


def positive_probabilities(fitted_pipeline: Pipeline, X: pd.DataFrame) -> np.ndarray:
    probabilities = fitted_pipeline.predict_proba(X)
    model = fitted_pipeline.named_steps["model"]
    classes = np.asarray(model.classes_)
    positive_positions = np.flatnonzero(classes == 1)
    if positive_positions.size != 1:
        raise ValueError(f"Fitted classifier classes do not contain a unique positive class: {classes.tolist()}")
    return np.asarray(probabilities[:, int(positive_positions[0])], dtype=float)

## 6. Train, tune thresholds, evaluate, explain, and save models

In [ ]:
results_rows: list[dict[str, Any]] = []
prediction_frames: list[pd.DataFrame] = []
latency_rows: list[dict[str, Any]] = []
coefficient_frames: list[pd.DataFrame] = []
tree_importance_frames: list[pd.DataFrame] = []
permutation_frames: list[pd.DataFrame] = []
threshold_records: dict[str, dict[str, Any]] = {}
trained_models: dict[tuple[str, str, str], Pipeline] = {}

split_strategies = {
    "chronological_primary": "chronological_split",
}
if grouped_sensitivity_available:
    split_strategies["aircraft_group_sensitivity"] = "aircraft_group_split"

for split_strategy, split_column in split_strategies.items():
    print(f"\n=== Split strategy: {split_strategy} ===")
    train_mask = split_assignments[split_column].eq("train")
    validation_mask = split_assignments[split_column].eq("validation")
    test_mask = split_assignments[split_column].eq("test")

    y_train = model_data.loc[train_mask, TARGET_COLUMN].astype(int)
    y_validation = model_data.loc[validation_mask, TARGET_COLUMN].astype(int)
    y_test = model_data.loc[test_mask, TARGET_COLUMN].astype(int)

    if not has_both_classes(y_train):
        raise ValueError(f"{split_strategy} training target does not contain both classes.")

    for feature_set_name, feature_specification in feature_sets.items():
        if not any(
            run_enabled(split_strategy, feature_set_name, specification.name)
            for specification in model_specifications
        ):
            continue

        numeric_candidates = list(feature_specification["numeric"])
        categorical_candidates = list(feature_specification["categorical"])
        candidate_columns = numeric_candidates + categorical_candidates

        X_train_candidates = model_data.loc[train_mask, candidate_columns].copy()
        usable_numeric, usable_categorical, dropped_columns = usable_columns_for_training(
            X_train_candidates,
            numeric_candidates,
            categorical_candidates,
        )
        usable_columns = usable_numeric + usable_categorical

        X_train = model_data.loc[train_mask, usable_columns].copy()
        X_validation = model_data.loc[validation_mask, usable_columns].copy()
        X_test = model_data.loc[test_mask, usable_columns].copy()

        print(
            f"{feature_set_name}: {len(usable_numeric)} numeric, "
            f"{len(usable_categorical)} categorical, {len(dropped_columns)} dropped"
        )

        for model_specification in model_specifications:
            if not run_enabled(
                split_strategy,
                feature_set_name,
                model_specification.name,
            ):
                continue

            run_key = f"{split_strategy}__{feature_set_name}__{model_specification.name}"
            print(f"  Training {run_key}")

            pipeline = build_pipeline(
                usable_numeric,
                usable_categorical,
                model_specification,
            )

            fit_start = time.perf_counter()
            pipeline.fit(X_train, y_train)
            fit_seconds = time.perf_counter() - fit_start

            validation_probabilities = positive_probabilities(pipeline, X_validation)
            threshold_selection = select_f1_threshold(y_validation, validation_probabilities)
            threshold = require_float(threshold_selection["threshold"], "selected threshold")
            threshold_policy = require_text(threshold_selection["policy"], "threshold policy")

            test_probabilities = positive_probabilities(pipeline, X_test)
            validation_metrics = classification_metrics(
                y_validation,
                validation_probabilities,
                threshold,
            )
            test_metrics = classification_metrics(
                y_test,
                test_probabilities,
                threshold,
            )

            latency = measure_single_event_latency_ms(
                pipeline,
                X_test,
                maximum_events=LATENCY_EVENT_SAMPLE_MAX,
                random_state=RANDOM_STATE,
            )

            transformed_count = len(transformed_feature_names(pipeline))
            result_row: dict[str, Any] = {
                "model_version": MODEL_VERSION,
                "target_mode": target_definition["mode"],
                "target_source_column": target_definition["source_column"],
                "target_source_threshold": target_definition.get("threshold"),
                "split_strategy": split_strategy,
                "feature_set": feature_set_name,
                "model": model_specification.name,
                "threshold_policy": threshold_policy,
                "selected_threshold": threshold,
                "train_rows": len(X_train),
                "validation_rows": len(X_validation),
                "test_rows": len(X_test),
                "input_numeric_features": len(usable_numeric),
                "input_categorical_features": len(usable_categorical),
                "input_total_features": len(usable_columns),
                "transformed_feature_count": transformed_count,
                "dropped_training_constant_or_all_missing_features": "|".join(dropped_columns),
                "fit_seconds": float(fit_seconds),
                **{f"validation_{key}": value for key, value in validation_metrics.items()},
                **{f"test_{key}": value for key, value in test_metrics.items()},
                **latency,
            }
            results_rows.append(result_row)

            latency_rows.append(
                {
                    "model_version": MODEL_VERSION,
                    "split_strategy": split_strategy,
                    "feature_set": feature_set_name,
                    "model": model_specification.name,
                    "measurement_type": "single_event_end_to_end_predict_proba",
                    **latency,
                }
            )

            threshold_records[run_key] = {
                "selected_threshold": threshold,
                "policy": threshold_policy,
                **threshold_selection,
            }

            prediction_frame = model_data.loc[
                test_mask,
                [ID_COLUMN, GROUP_COLUMN, TIME_COLUMN, RAW_TARGET_COLUMN, TARGET_COLUMN],
            ].copy()
            prediction_frame["model_version"] = MODEL_VERSION
            prediction_frame["target_mode"] = target_definition["mode"]
            prediction_frame["split_strategy"] = split_strategy
            prediction_frame["feature_set"] = feature_set_name
            prediction_frame["model"] = model_specification.name
            prediction_frame["predicted_probability"] = test_probabilities
            prediction_frame["selected_threshold"] = threshold
            prediction_frame["predicted_label"] = (test_probabilities >= threshold).astype("int8")
            prediction_frames.append(prediction_frame)

            model_filename = (
                f"{slugify(split_strategy)}__{feature_set_name}__{model_specification.name}.joblib"
            )
            model_path = MODELS_DIR / model_filename
            joblib.dump(pipeline, model_path)
            trained_models[(split_strategy, feature_set_name, model_specification.name)] = pipeline

            feature_names = transformed_feature_names(pipeline)
            fitted_estimator = pipeline.named_steps["model"]

            if model_specification.name == "logistic_regression":
                coefficients = np.asarray(fitted_estimator.coef_).reshape(-1)
                coefficient_frames.append(
                    pd.DataFrame(
                        {
                            "model_version": MODEL_VERSION,
                            "split_strategy": split_strategy,
                            "feature_set": feature_set_name,
                            "model": model_specification.name,
                            "transformed_feature": feature_names,
                            "coefficient": coefficients,
                            "absolute_coefficient": np.abs(coefficients),
                        }
                    ).sort_values("absolute_coefficient", ascending=False)
                )

            if model_specification.name in {"decision_tree", "random_forest"}:
                importances = np.asarray(fitted_estimator.feature_importances_, dtype=float)
                tree_importance_frames.append(
                    pd.DataFrame(
                        {
                            "model_version": MODEL_VERSION,
                            "split_strategy": split_strategy,
                            "feature_set": feature_set_name,
                            "model": model_specification.name,
                            "transformed_feature": feature_names,
                            "importance": importances,
                        }
                    ).sort_values("importance", ascending=False)
                )

            if model_specification.name == "decision_tree":
                rules = export_text(
                    fitted_estimator,
                    feature_names=feature_names.tolist(),
                    max_depth=5,
                    decimals=4,
                )
                rules_path = MODELS_DIR / (
                    f"{slugify(split_strategy)}__{feature_set_name}__decision_tree_rules.txt"
                )
                rules_path.write_text(rules, encoding="utf-8")

            if (
                RUN_PERMUTATION_IMPORTANCE
                and split_strategy == "chronological_primary"
                and feature_set_name == "fused"
                and model_specification.name == "random_forest"
                and has_both_classes(y_test)
            ):
                sample_positions = stratified_sample_positions(
                    y_test.reset_index(drop=True),
                    maximum=PERMUTATION_SAMPLE_MAX,
                    random_state=RANDOM_STATE,
                )
                X_permutation = X_test.reset_index(drop=True).iloc[sample_positions]
                y_permutation = y_test.reset_index(drop=True).iloc[sample_positions]
                if has_both_classes(y_permutation):
                    permutation_result = cast(
                        Any,
                        permutation_importance(
                            pipeline,
                            X_permutation,
                            y_permutation,
                            scoring="average_precision",
                            n_repeats=PERMUTATION_REPEATS,
                            random_state=RANDOM_STATE,
                            n_jobs=-1,
                        ),
                    )
                    importance_mean = np.asarray(
                        permutation_result.importances_mean,
                        dtype=float,
                    )
                    importance_std = np.asarray(
                        permutation_result.importances_std,
                        dtype=float,
                    )
                    permutation_frames.append(
                        pd.DataFrame(
                            {
                                "model_version": MODEL_VERSION,
                                "split_strategy": split_strategy,
                                "feature_set": feature_set_name,
                                "model": model_specification.name,
                                "input_feature": usable_columns,
                                "importance_mean": importance_mean,
                                "importance_std": importance_std,
                                "sample_rows": len(X_permutation),
                                "repeats": PERMUTATION_REPEATS,
                            }
                        ).sort_values("importance_mean", ascending=False)
                    )

results = pd.DataFrame(results_rows).sort_values(
    ["split_strategy", "feature_set", "model"]
).reset_index(drop=True)
predictions = pd.concat(prediction_frames, ignore_index=True)
latency_results = pd.DataFrame(latency_rows)
logistic_coefficients = (
    pd.concat(coefficient_frames, ignore_index=True)
    if coefficient_frames
    else pd.DataFrame()
)
tree_feature_importance = (
    pd.concat(tree_importance_frames, ignore_index=True)
    if tree_importance_frames
    else pd.DataFrame()
)
permutation_results = (
    pd.concat(permutation_frames, ignore_index=True)
    if permutation_frames
    else pd.DataFrame()
)

results.to_csv(RESULTS_OUT, index=False)
predictions.to_csv(PREDICTIONS_OUT, index=False)
latency_results.to_csv(LATENCY_OUT, index=False)
logistic_coefficients.to_csv(LOGISTIC_COEFFICIENTS_OUT, index=False)
tree_feature_importance.to_csv(TREE_IMPORTANCE_OUT, index=False)
permutation_results.to_csv(PERMUTATION_IMPORTANCE_OUT, index=False)
THRESHOLDS_OUT.write_text(json.dumps(threshold_records, indent=2), encoding="utf-8")

display(
    results[
        [
            "split_strategy",
            "feature_set",
            "model",
            "selected_threshold",
            "validation_auprc",
            "test_auprc",
            "test_f1",
            "test_recall",
            "test_precision",
            "latency_p95_ms",
        ]
    ]
)


## 7. Compare fused performance with the strongest unimodal baseline

In [ ]:
fused_gain_rows: list[dict[str, Any]] = []
for split_strategy in sorted(results["split_strategy"].unique().tolist()):
    strategy_results = results.loc[results["split_strategy"].eq(split_strategy)]
    for model_name in sorted(strategy_results["model"].unique().tolist()):
        model_results = strategy_results.loc[strategy_results["model"].eq(model_name)]
        fused_rows = model_results.loc[model_results["feature_set"].eq("fused")]
        unimodal_rows = model_results.loc[
            model_results["feature_set"].isin(["physical", "cyber"])
        ].copy()
        if len(fused_rows) != 1 or unimodal_rows.empty:
            continue

        fused_auprc = optional_float(fused_rows.iloc[0]["test_auprc"])
        unimodal_rows["_numeric_test_auprc"] = pd.to_numeric(
            unimodal_rows["test_auprc"], errors="coerce"
        )
        valid_unimodal = unimodal_rows.dropna(subset=["_numeric_test_auprc"])
        if not math.isfinite(fused_auprc) or valid_unimodal.empty:
            continue

        best_unimodal_position = int(
            np.nanargmax(valid_unimodal["_numeric_test_auprc"].to_numpy(dtype=float))
        )
        best_unimodal = valid_unimodal.iloc[best_unimodal_position]
        best_unimodal_auprc = require_float(
            best_unimodal["_numeric_test_auprc"],
            "best unimodal test AUPRC",
        )

        fused_gain_rows.append(
            {
                "model_version": MODEL_VERSION,
                "split_strategy": split_strategy,
                "model": model_name,
                "fused_test_auprc": fused_auprc,
                "best_unimodal_feature_set": require_text(
                    best_unimodal["feature_set"],
                    "best unimodal feature set",
                ),
                "best_unimodal_test_auprc": best_unimodal_auprc,
                "fused_minus_best_unimodal_auprc": fused_auprc - best_unimodal_auprc,
            }
        )

fused_gain_summary = pd.DataFrame(fused_gain_rows)
fused_gain_summary.to_csv(FUSED_GAIN_OUT, index=False)
display(fused_gain_summary)


## 8. Generate diagnostic and explainability figures

In [ ]:
chronological_predictions = predictions.loc[
    predictions["split_strategy"].eq("chronological_primary")
].copy()

# Precision-recall curves: one figure per feature set when both test classes exist.
for feature_set_name in ["physical", "cyber", "fused"]:
    feature_predictions = chronological_predictions.loc[
        chronological_predictions["feature_set"].eq(feature_set_name)
    ]
    if feature_predictions.empty or not has_both_classes(feature_predictions[TARGET_COLUMN]):
        continue

    plt.figure(figsize=(8, 6))
    for model_name in sorted(feature_predictions["model"].unique().tolist()):
        model_predictions = feature_predictions.loc[
            feature_predictions["model"].eq(model_name)
        ]
        precision_values, recall_values, _ = precision_recall_curve(
            model_predictions[TARGET_COLUMN],
            model_predictions["predicted_probability"],
        )
        auprc = safe_average_precision(
            model_predictions[TARGET_COLUMN],
            model_predictions["predicted_probability"].to_numpy(dtype=float),
        )
        plt.plot(recall_values, precision_values, label=f"{model_name} (AUPRC={auprc:.3f})")
    prevalence = require_float(
        feature_predictions[TARGET_COLUMN].mean(),
        f"{feature_set_name} test prevalence",
    )
    plt.axhline(prevalence, linestyle="--", label=f"Prevalence={prevalence:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Test precision-recall curves — {feature_set_name}")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"precision_recall_{feature_set_name}.png", dpi=180)
    plt.close()

# Calibration curves are skipped for single-class test partitions.
for feature_set_name in ["physical", "cyber", "fused"]:
    feature_predictions = chronological_predictions.loc[
        chronological_predictions["feature_set"].eq(feature_set_name)
    ]
    if feature_predictions.empty or not has_both_classes(feature_predictions[TARGET_COLUMN]):
        continue

    plt.figure(figsize=(8, 6))
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    for model_name in sorted(feature_predictions["model"].unique().tolist()):
        model_predictions = feature_predictions.loc[
            feature_predictions["model"].eq(model_name)
        ]
        fraction_positive, mean_predicted = calibration_curve(
            model_predictions[TARGET_COLUMN],
            model_predictions["predicted_probability"],
            n_bins=10,
            strategy="quantile",
        )
        plt.plot(mean_predicted, fraction_positive, marker="o", label=model_name)
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed positive fraction")
    plt.title(f"Test calibration — {feature_set_name}")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"calibration_{feature_set_name}.png", dpi=180)
    plt.close()

# One confusion-matrix figure per feature-set/model pair; no subplots.
for feature_set_name in ["physical", "cyber", "fused"]:
    for model_name in sorted(chronological_predictions["model"].unique().tolist()):
        subset = chronological_predictions.loc[
            chronological_predictions["feature_set"].eq(feature_set_name)
            & chronological_predictions["model"].eq(model_name)
        ]
        if subset.empty:
            continue
        matrix = confusion_matrix(
            subset[TARGET_COLUMN],
            subset["predicted_label"],
            labels=[0, 1],
        )
        plt.figure(figsize=(5, 4))
        plt.imshow(matrix)
        plt.xticks([0, 1], ["Predicted 0", "Predicted 1"])
        plt.yticks([0, 1], ["Actual 0", "Actual 1"])
        for row_index in range(2):
            for column_index in range(2):
                plt.text(
                    column_index,
                    row_index,
                    str(matrix[row_index, column_index]),
                    ha="center",
                    va="center",
                )
        plt.title(f"Confusion matrix — {feature_set_name} / {model_name}")
        plt.tight_layout()
        plt.savefig(
            FIGURES_DIR / f"confusion_{feature_set_name}_{model_name}.png",
            dpi=180,
        )
        plt.close()

if not logistic_coefficients.empty:
    fused_logistic = logistic_coefficients.loc[
        logistic_coefficients["split_strategy"].eq("chronological_primary")
        & logistic_coefficients["feature_set"].eq("fused")
    ].nlargest(30, "absolute_coefficient")
    if not fused_logistic.empty:
        ordered = fused_logistic.sort_values("coefficient")
        plt.figure(figsize=(10, 9))
        plt.barh(ordered["transformed_feature"], ordered["coefficient"])
        plt.xlabel("Standardized logistic coefficient")
        plt.ylabel("Transformed feature")
        plt.title("Fused logistic regression — strongest coefficients")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "fused_logistic_coefficients.png", dpi=180)
        plt.close()

if not tree_feature_importance.empty:
    fused_forest = tree_feature_importance.loc[
        tree_feature_importance["split_strategy"].eq("chronological_primary")
        & tree_feature_importance["feature_set"].eq("fused")
        & tree_feature_importance["model"].eq("random_forest")
    ].nlargest(30, "importance")
    if not fused_forest.empty:
        ordered = fused_forest.sort_values("importance")
        plt.figure(figsize=(10, 9))
        plt.barh(ordered["transformed_feature"], ordered["importance"])
        plt.xlabel("Random-forest impurity importance")
        plt.ylabel("Transformed feature")
        plt.title("Fused random forest — top transformed features")
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "fused_random_forest_importance.png", dpi=180)
        plt.close()

fused_tree_key = ("chronological_primary", "fused", "decision_tree")
if fused_tree_key in trained_models:
    fused_tree_pipeline = trained_models[fused_tree_key]
    fused_tree_model = fused_tree_pipeline.named_steps["model"]
    fused_tree_names = transformed_feature_names(fused_tree_pipeline)
    plt.figure(figsize=(24, 12))
    plot_tree(
        fused_tree_model,
        feature_names=fused_tree_names.tolist(),
        class_names=["0", "1"],
        filled=False,
        rounded=True,
        proportion=True,
        max_depth=5,
        fontsize=7,
    )
    plt.title("Fused shallow decision tree — chronological split")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fused_decision_tree.png", dpi=180)
    plt.close()

print(f"Figures written to: {FIGURES_DIR}")


## 9. Build summary, data dictionary, checksums, and final regression checks

In [ ]:
def output_dictionary_rows() -> list[dict[str, str]]:
    descriptions = {
        SPLITS_OUT.name: "Event-level chronological and optional aircraft-group split assignments, including raw and model targets.",
        RESULTS_OUT.name: "Validation/test metrics, thresholds, feature counts, fit time, and latency for every model run.",
        PREDICTIONS_OUT.name: "Event-level held-out test probabilities and thresholded predictions.",
        THRESHOLDS_OUT.name: "Validation-selected thresholds and threshold-selection metadata.",
        TARGET_DEFINITION_OUT.name: "Frozen model-target source, operator, threshold, prevalence, and fallback rationale.",
        LOGISTIC_COEFFICIENTS_OUT.name: "Expanded standardized logistic-regression coefficients.",
        TREE_IMPORTANCE_OUT.name: "Expanded tree and random-forest impurity feature importances.",
        PERMUTATION_IMPORTANCE_OUT.name: "Input-column permutation importance for chronological fused random forest when test classes permit it.",
        LATENCY_OUT.name: "Single-event end-to-end predict_proba latency measurements summarized by model run.",
        FUSED_GAIN_OUT.name: "Fused test AUPRC minus the strongest unimodal test AUPRC when those metrics are defined.",
        FEATURE_MANIFEST_OUT.name: "Frozen feature-set definitions and exclusion policies.",
    }
    return [
        {"artifact": path.name, "artifact_role": descriptions[path.name]}
        for path in [
            SPLITS_OUT,
            RESULTS_OUT,
            PREDICTIONS_OUT,
            THRESHOLDS_OUT,
            TARGET_DEFINITION_OUT,
            LOGISTIC_COEFFICIENTS_OUT,
            TREE_IMPORTANCE_OUT,
            PERMUTATION_IMPORTANCE_OUT,
            LATENCY_OUT,
            FUSED_GAIN_OUT,
            FEATURE_MANIFEST_OUT,
        ]
    ]


expected_primary_runs = len(feature_sets) * len(model_specifications)
actual_primary_runs = require_int(
    results["split_strategy"].eq("chronological_primary").sum(),
    "chronological model run count",
)
if actual_primary_runs != expected_primary_runs:
    raise AssertionError(
        f"Expected {expected_primary_runs} chronological model runs; found {actual_primary_runs}."
    )

expected_prediction_rows = 0
for split_strategy, split_column in split_strategies.items():
    test_rows = require_int(
        split_assignments[split_column].eq("test").sum(),
        f"{split_strategy} test rows",
    )
    enabled_runs = sum(
        1
        for feature_set_name in feature_sets
        for model_specification in model_specifications
        if run_enabled(split_strategy, feature_set_name, model_specification.name)
    )
    expected_prediction_rows += test_rows * enabled_runs
if len(predictions) != expected_prediction_rows:
    raise AssertionError(
        f"Expected {expected_prediction_rows:,} prediction rows; found {len(predictions):,}."
    )

if predictions["predicted_probability"].isna().any():
    raise AssertionError("Test predictions contain missing probabilities.")
if not predictions["predicted_probability"].between(0, 1).all():
    raise AssertionError("Test probabilities fall outside [0, 1].")
if results["selected_threshold"].isna().any():
    raise AssertionError("At least one model run lacks a selected threshold.")
if not results["selected_threshold"].between(0, 1).all():
    raise AssertionError("At least one selected threshold falls outside [0, 1].")

fused_latency_passes = results.loc[
    results["split_strategy"].eq("chronological_primary")
    & results["feature_set"].eq("fused"),
    ["model", "latency_p95_ms"],
].copy()
fused_latency_passes["p95_under_500ms"] = fused_latency_passes["latency_p95_ms"].le(500.0)

summary = {
    "definitions": {
        "model_version": MODEL_VERSION,
        "preferred_raw_target": RAW_TARGET_COLUMN,
        "actual_model_target": TARGET_COLUMN,
        "target_definition": target_definition,
        "primary_window": PRIMARY_WINDOW_NAME,
        "feature_sets": list(feature_sets.keys()),
        "models": [specification.name for specification in model_specifications],
        "chronological_split_fractions": {
            "train": TRAIN_FRACTION,
            "validation": VALIDATION_FRACTION,
            "test": TEST_FRACTION,
        },
        "grouped_sensitivity_requested": RUN_GROUPED_SENSITIVITY,
        "grouped_sensitivity_available": grouped_sensitivity_available,
        "grouped_split_seed_used": grouped_seed,
        "calendar_features_included": INCLUDE_CALENDAR_FEATURES,
    },
    "interpretation": {
        "site_coincident_claim": False,
        "causal_claim": False,
        "statement": (
            "Results quantify predictive performance within the Notebook 03 alignment design. "
            "Controlled replay results must not be interpreted as real simultaneous PANC aircraft/cyber observations."
        ),
    },
    "inputs": {
        "feature_file": str(FEATURES_IN.relative_to(PROJECT_ROOT)),
        "target_file": str(TARGETS_IN.relative_to(PROJECT_ROOT)),
        "event_rows": len(model_data),
        "unique_aircraft_keys": require_int(
            model_data[GROUP_COLUMN].nunique(),
            "unique aircraft keys",
        ),
        "raw_positive_targets": require_int(
            model_data[RAW_TARGET_COLUMN].sum(),
            "raw positive targets",
        ),
        "model_positive_targets": require_int(
            model_data[TARGET_COLUMN].sum(),
            "model positive targets",
        ),
        "model_target_prevalence": require_float(
            model_data[TARGET_COLUMN].mean(),
            "model target prevalence",
        ),
        "event_time_start_utc": model_data[TIME_COLUMN].min().isoformat(),
        "event_time_end_utc": model_data[TIME_COLUMN].max().isoformat(),
        "input_sha256": {
            "feature_file": sha256_file(FEATURES_IN),
            "target_file": sha256_file(TARGETS_IN),
        },
    },
    "features": {
        name: {
            "numeric_candidates": specification["numeric"],
            "categorical_candidates": specification["categorical"],
        }
        for name, specification in feature_sets.items()
    },
    "splits": json.loads(split_summary.to_json(orient="records", date_format="iso")),
    "outputs": {
        "model_runs": len(results),
        "test_prediction_rows": len(predictions),
        "saved_model_files": len(list(MODELS_DIR.glob("*.joblib"))),
        "saved_figure_files": len(list(FIGURES_DIR.glob("*.png"))),
        "chronological_fused_latency": fused_latency_passes.to_dict("records"),
        "chronological_fused_gain": (
            fused_gain_summary.loc[
                fused_gain_summary.get("split_strategy", pd.Series(dtype="string")).eq(
                    "chronological_primary"
                )
            ].to_dict("records")
            if not fused_gain_summary.empty
            else []
        ),
    },
    "environment": {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
    "regression_checks": [
        {
            "metric": "input_event_rows_preserved_in_split_assignments",
            "expected": len(model_data),
            "actual": len(split_assignments),
            "match": len(model_data) == len(split_assignments),
        },
        {
            "metric": "chronological_training_has_both_classes",
            "expected": True,
            "actual": has_both_classes(
                split_assignments.loc[
                    split_assignments["chronological_split"].eq("train"),
                    TARGET_COLUMN,
                ]
            ),
            "match": has_both_classes(
                split_assignments.loc[
                    split_assignments["chronological_split"].eq("train"),
                    TARGET_COLUMN,
                ]
            ),
        },
        {
            "metric": "chronological_model_runs",
            "expected": expected_primary_runs,
            "actual": actual_primary_runs,
            "match": expected_primary_runs == actual_primary_runs,
        },
        {
            "metric": "prediction_rows",
            "expected": expected_prediction_rows,
            "actual": len(predictions),
            "match": expected_prediction_rows == len(predictions),
        },
        {
            "metric": "target_leakage_columns_in_predictors",
            "expected": 0,
            "actual": len(leakage_intersection),
            "match": len(leakage_intersection) == 0,
        },
        {
            "metric": "identifier_columns_in_predictors",
            "expected": 0,
            "actual": len(identifier_intersection),
            "match": len(identifier_intersection) == 0,
        },
    ],
}
SUMMARY_OUT.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")

pd.DataFrame(output_dictionary_rows()).to_csv(DATA_DICTIONARY_OUT, index=False)

artifact_paths = [
    SPLITS_OUT,
    RESULTS_OUT,
    PREDICTIONS_OUT,
    THRESHOLDS_OUT,
    TARGET_DEFINITION_OUT,
    LOGISTIC_COEFFICIENTS_OUT,
    TREE_IMPORTANCE_OUT,
    PERMUTATION_IMPORTANCE_OUT,
    LATENCY_OUT,
    FUSED_GAIN_OUT,
    FEATURE_MANIFEST_OUT,
    SUMMARY_OUT,
    DATA_DICTIONARY_OUT,
]
artifact_paths.extend(sorted(MODELS_DIR.glob("*")))
artifact_paths.extend(sorted(FIGURES_DIR.glob("*.png")))

checksum_lines = []
for path in artifact_paths:
    if path.is_file() and path != CHECKSUMS_OUT:
        checksum_lines.append(
            f"{sha256_file(path)}  {path.relative_to(OUTPUT_DIR).as_posix()}"
        )
CHECKSUMS_OUT.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

final_checks = pd.DataFrame(summary["regression_checks"])
display(final_checks)
display(fused_latency_passes)

if not final_checks["match"].all():
    raise AssertionError("One or more final regression checks failed.")

print("\nNotebook 04 completed successfully.")
print(f"Target mode:         {target_definition['mode']}")
print(f"Target source:       {target_definition['source_column']}")
print(f"Model runs:          {len(results):,}")
print(f"Prediction rows:     {len(predictions):,}")
print(f"Saved model files:   {len(list(MODELS_DIR.glob('*.joblib'))):,}")
print(f"Saved figure files:  {len(list(FIGURES_DIR.glob('*.png'))):,}")
print(f"Output directory:    {OUTPUT_DIR}")


## Output interpretation

The notebook records whether it used the raw any-attack target or the training-only high-attack-burden fallback. All labels and label-derived burden fields remain excluded from predictors. Results describe controlled replay performance only; they do not establish site-coincident or causal aircraft–cyber relationships.
